# สร้าง Dataset กลางและแบ่งชุดข้อมูล

Notebook นี้รวมโค้ดที่ใช้ตรวจผล Harmonization, รวม NSL-KDD/UNSW-NB15 และสร้าง Train/Validation/Test ไว้ในไฟล์เดียว เปิดอ่านและรันทีละส่วนได้ โดยไม่ต้อง import ฟังก์ชันจากไฟล์ Python ภายนอก

การแบ่งชุดและการสุ่ม Train **ไม่ใช้ label**; `labels` และ `metadata` จะยังถูกเก็บแยกจาก `features`

## 1. เตรียมเครื่องมือและหาโฟลเดอร์โปรเจกต์

ส่วนนี้หา path จากตำแหน่งที่เปิด notebook และเตรียม checksum/การตรวจตารางไว้ที่นี่ เพื่อให้ notebook ใช้งานได้ด้วยตัวเอง ไม่ขึ้นกับตำแหน่งติดตั้งบนเครื่องใดเครื่องหนึ่ง.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd


def find_project_root(start=None):
    """หาโฟลเดอร์โปรเจกต์จากตำแหน่งที่เปิด notebook โดยไม่ล็อก path เครื่องใดเครื่องหนึ่ง."""
    origin = Path(start or Path.cwd()).resolve()
    for candidate in (origin, *origin.parents):
        has_pipeline = (candidate / "data-pipeline").is_dir()
        has_project_marker = (candidate / ".git").exists() or (candidate / "README.md").is_file()
        if has_pipeline and has_project_marker:
            return candidate
    raise FileNotFoundError(f"หาโฟลเดอร์โปรเจกต์จากตำแหน่งนี้ไม่พบ: {origin}")


PROJECT_ROOT = find_project_root()
PIPELINE_ROOT = PROJECT_ROOT / "data-pipeline"
COMMON_ROOT = PIPELINE_ROOT / "common"
OUTPUT_ROOT = COMMON_ROOT / "output"


def dataset_dir(dataset_name):
    """คืน path ของ dataset ที่โครงการนี้รองรับ."""
    known_datasets = {"NSL-KDD", "UNSW-NB15"}
    if dataset_name not in known_datasets:
        raise ValueError(f"ไม่รู้จัก dataset {dataset_name!r}; รองรับ {sorted(known_datasets)}")
    return PIPELINE_ROOT / dataset_name


def sha256_file(path, chunk_size=1024 * 1024):
    """คำนวณ checksum เป็นชิ้น ๆ เพื่อไม่ต้องโหลดไฟล์ใหญ่ทั้งหมดเข้า memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


KEY = "record_id"
FORBIDDEN_FEATURE_COLUMNS = {
    "label", "label_raw", "attack_category", "label_binary", "is_attack",
    "source_dataset", "orig_row_id", "capture_time", "split_group",
    "sample_weight", "dedup_flag", "data_split", "difficulty_level",
}


def validate_aligned_tables(features, labels, metadata):
    """ตรวจว่า 3 ตารางมี record_id และลำดับตรงกัน พร้อมกัน label/metadata รั่วเข้า X."""
    tables = {"features": features, "labels": labels, "metadata": metadata}
    for table_name, frame in tables.items():
        if KEY not in frame.columns:
            raise ValueError(f"ตาราง {table_name} ไม่มีคอลัมน์ {KEY!r}")
        if frame[KEY].isna().any() or frame[KEY].astype(str).eq("").any():
            raise ValueError(f"{table_name}.{KEY} มีค่าว่าง")
        if frame[KEY].duplicated().any():
            examples = frame.loc[frame[KEY].duplicated(), KEY].head(3).tolist()
            raise ValueError(f"{table_name}.{KEY} ซ้ำ ตัวอย่าง: {examples}")

    reference_ids = features[KEY].astype(str).reset_index(drop=True)
    for table_name, frame in (("labels", labels), ("metadata", metadata)):
        current_ids = frame[KEY].astype(str).reset_index(drop=True)
        if not reference_ids.equals(current_ids):
            raise ValueError(f"ลำดับ record_id ใน features และ {table_name} ไม่ตรงกัน")

    leaked_columns = FORBIDDEN_FEATURE_COLUMNS.intersection(features.columns)
    if leaked_columns:
        raise ValueError(f"พบ label/metadata ปะปนใน features: {sorted(leaked_columns)}")
    if set(labels.columns) == {KEY}:
        raise ValueError("ตาราง labels ไม่มีคอลัมน์ label")
    return {name: len(frame) for name, frame in tables.items()}


print(f"โฟลเดอร์โปรเจกต์: {PROJECT_ROOT}")
print(f"โฟลเดอร์ผลลัพธ์: {OUTPUT_ROOT}")


## 2. ตรวจสอบแหล่งข้อมูลและความถูกต้องของผล Harmonization

ก่อนรวมข้อมูลจะตรวจไฟล์ต้นทางด้วย SHA-256 และเลขรุ่นของ registry/feature contract ถ้าข้อมูลดิบเปลี่ยนไปหลัง Harmonization ระบบจะหยุดให้ทำขั้น Harmonization ใหม่ แทนการนำผลเก่ามาปนกับ raw รุ่นใหม่.

In [ ]:
DATASETS = ("NSL-KDD", "UNSW-NB15")
SPLITS = ("train", "validation", "test")
CORE_CANDIDATES = ("duration", "protocol", "bytes_forward", "bytes_backward")

def _verify_provenance(name: str, folder: Path) -> str | None:
    manifest_path = folder / "source_manifest.csv"
    provenance_path = folder / "provenance.json"
    if not manifest_path.is_file() or not provenance_path.is_file():
        return "Missing source manifest/provenance; rerun harmonization before including this source."
    provenance = json.loads(provenance_path.read_text(encoding="utf-8"))
    registry = json.loads((PIPELINE_ROOT / "common" / "dataset_registry.json").read_text(encoding="utf-8"))
    contract = json.loads((PIPELINE_ROOT / "common" / "feature_contract.json").read_text(encoding="utf-8"))
    expected_versions = (registry["datasets"][name]["version"], registry["registry_version"], contract["contract_version"])
    saved_versions = (provenance.get("source_version"), provenance.get("registry_version"), provenance.get("feature_contract_version"))
    if expected_versions != saved_versions:
        raise ValueError(f"{name}: source/registry/contract version changed since harmonization; rerun harmonization.")
    raw_dir = dataset_dir(name) / "raw"
    current_files = sorted(path for path in raw_dir.rglob("*")
                           if path.is_file() and path.suffix.lower() in {".csv", ".gz", ".txt"})
    current = {path.relative_to(raw_dir).as_posix(): sha256_file(path) for path in current_files}
    manifest = pd.read_csv(manifest_path)
    saved = dict(zip(manifest["file"].astype(str), manifest["sha256"].astype(str)))
    if current != saved:
        raise ValueError(f"{name}: raw files or checksums differ from the harmonized snapshot; rerun cleansing/harmonization before building.")
    return None

## 3. อ่านเฉพาะชุดที่พร้อมและผ่านการตรวจ

แต่ละแหล่งต้องมี `features.csv`, `labels.csv` และ `metadata.csv` ครบ พร้อม manifest/provenance ที่สอดคล้องกัน หากชุดใดยังไม่พร้อมจะบันทึกเหตุผลและข้ามชุดนั้น แต่ถ้าไม่มีชุดใดพร้อมจะหยุดทำงาน.

In [ ]:
def _read_ready_sources() -> tuple[list[pd.DataFrame], list[pd.DataFrame], list[pd.DataFrame], list[str], list[dict]]:
    feature_frames, label_frames, metadata_frames, included, readiness = [], [], [], [], []
    for name in DATASETS:
        folder = dataset_dir(name) / "output" / "harmonized"
        paths = [folder / f"{part}.csv" for part in ("features", "labels", "metadata")]
        if not all(path.is_file() for path in paths):
            readiness.append({"dataset": name, "ready": False, "reason": "Missing local harmonized table(s); source not included."})
            continue
        provenance_issue = _verify_provenance(name, folder)
        if provenance_issue:
            readiness.append({"dataset": name, "ready": False, "reason": provenance_issue})
            continue
        features, labels, metadata = [pd.read_csv(path) for path in paths]
        validate_aligned_tables(features, labels, metadata)
        if not metadata["source_dataset"].eq(name).all():
            raise ValueError(f"{name}: metadata source_dataset contains mixed or incorrect dataset names.")
        feature_frames.append(features)
        label_frames.append(labels)
        metadata_frames.append(metadata)
        included.append(name)
        readiness.append({"dataset": name, "ready": True, "reason": f"Loaded {len(features):,} verified harmonized rows."})
    if not included:
        raise FileNotFoundError("No dataset has complete harmonized CSV tables. Run each available harmonization notebook first.")
    return feature_frames, label_frames, metadata_frames, included, readiness

## 4. แบ่ง Train / Validation / Test

คง official Test ไว้ และกัน Validation จาก official Train ด้วย seed คงที่ ไม่ใช้ label ในการแบ่ง เพื่อไม่ให้คำตอบจริงมีอิทธิพลต่อการสุ่มชุดข้อมูล.

In [ ]:
def assign_splits(metadata: pd.DataFrame, seed: int = 42, validation_fraction: float = 0.20) -> pd.Series:
    """Preserve official train/test for NSL-KDD and UNSW-NB15; carve validation from train."""
    result = pd.Series(pd.NA, index=metadata.index, dtype="string")
    for source, indexes in metadata.groupby("source_dataset", sort=True).groups.items():
        block = metadata.loc[indexes]
        if source in {"NSL-KDD", "UNSW-NB15"}:
            official = block["source_split"].astype("string").str.lower()
            if not official.isin(["train", "test"]).all():
                raise ValueError(f"{source}: expected official train/test markers; found {sorted(official.dropna().unique())}.")
            train_ids = block.index[official.eq("train")].to_numpy()
            test_ids = block.index[official.eq("test")].to_numpy()
            if len(train_ids) < 2 or not len(test_ids):
                raise ValueError(f"{source}: official train/test files must both contain records.")
            # Deliberately independent of labels: avoid label-based training subsampling/splitting.
            rng = np.random.default_rng(seed)
            shuffled = rng.permutation(train_ids)
            n_validation = max(1, int(round(len(shuffled) * validation_fraction)))
            if n_validation >= len(shuffled):
                n_validation = len(shuffled) - 1
            validation_ids = shuffled[:n_validation]
            new_train_ids = shuffled[n_validation:]
            result.loc[new_train_ids] = "train"
            result.loc[validation_ids] = "validation"
            result.loc[test_ids] = "test"
        else:
            raise ValueError(f"Unsupported source {source!r}; this pipeline supports NSL-KDD and UNSW-NB15 only.")
    if result.isna().any():
        raise AssertionError("At least one record did not receive a split.")
    return result

def _write_triplet(folder: Path, features: pd.DataFrame, labels: pd.DataFrame, metadata: pd.DataFrame) -> None:
    folder.mkdir(parents=True, exist_ok=True)
    for name, frame in (("features", features), ("labels", labels), ("metadata", metadata)):
        frame.to_csv(folder / f"{name}.csv", index=False, encoding="utf-8")

## 5. ตรวจว่า feature ใช้ได้กับ dataset ที่มีอยู่หรือไม่

รายงานแยกให้เห็นว่า feature ใดครบในข้อมูลที่พร้อมปัจจุบัน และ feature ใดได้รับการยืนยันว่าครบทั้ง NSL-KDD และ UNSW-NB15 ถ้ามีแหล่งเดียวพร้อม จะระบุเป็นการตรวจเฉพาะแหล่ง ไม่สรุปว่าใช้ร่วมกันได้แล้ว.

In [ ]:
def _feature_compatibility(features: pd.DataFrame, metadata: pd.DataFrame,
                           included: list[str], readiness: list[dict]) -> list[dict]:
    """Distinguish source-local usability from verified compatibility across both configured datasets."""
    ready_names = {row["dataset"] for row in readiness if row["ready"]}
    statuses = []
    for candidate in CORE_CANDIDATES:
        source_checks = {}
        for source in DATASETS:
            source_features = features.loc[metadata["source_dataset"].eq(source)]
            ready = source in ready_names
            available = ready and candidate in source_features and source_features[candidate].notna().any()
            complete = ready and candidate in source_features and source_features[candidate].notna().all()
            source_checks[source] = {"ready": ready, "available": bool(available), "complete": bool(complete)}
        usable_here = bool(included) and all(source_checks[source]["complete"] for source in included)
        joint_ready = set(included) == set(DATASETS) and all(source_checks[source]["complete"] for source in DATASETS)
        missing_sources = sorted(set(DATASETS) - ready_names)
        if missing_sources:
            reason = "Configured dataset(s) not ready: " + ", ".join(missing_sources)
        elif not joint_ready:
            reason = "Feature is missing, incomplete, or unverified in at least one configured source"
        else:
            reason = "Verified and complete in both configured datasets"
        statuses.append({
            "feature": candidate,
            "usable_in_current_build": usable_here,
            "eligible_for_joint_profile": joint_ready,
            "source_checks": source_checks,
            "reason": reason,
        })
    return statuses

## 6. สุ่ม Train ให้สมดุลตามแหล่งข้อมูล

เมื่อมีหลาย dataset จะสุ่มจำนวน record เท่ากันจากแต่ละแหล่ง โดยไม่ใช้ label ส่วน Train เต็มยังเก็บไว้ใน `train_full`; `sample_weight` เป็น metadata ไม่ใช่ feature สำหรับโมเดล.

In [ ]:
def _sample_training_equally_by_source(
    full_train: tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame],
    seed: int,
) -> tuple[tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame], int]:
    """สุ่มจำนวนแถว Train เท่ากันต่อ dataset โดยสุ่มอิสระจาก label.

    คืนทั้งตาราง features/labels/metadata ที่ยังจัดแนวกัน และจำนวนแถวต่อแหล่ง
    ตาราง Train เต็มยังถูกเก็บแยกไว้ ไม่ได้ถูกลบทิ้งจาก output
    """
    train_features, train_labels, train_metadata = full_train
    rows_per_source = train_metadata["source_dataset"].value_counts().sort_index()
    sample_size = int(rows_per_source.min())

    selected_indexes = []
    for source in sorted(rows_per_source.index):
        source_indexes = train_metadata.index[train_metadata["source_dataset"].eq(source)]
        selected_indexes.extend(
            pd.Series(source_indexes).sample(n=sample_size, random_state=seed).tolist()
        )
    selected_indexes.sort()

    sampled_features = train_features.loc[selected_indexes].reset_index(drop=True)
    sampled_labels = train_labels.loc[selected_indexes].reset_index(drop=True)
    sampled_metadata = train_metadata.loc[selected_indexes].reset_index(drop=True)

    # sample_weight เป็น metadata เท่านั้น ไม่ใช่คอลัมน์สำหรับป้อนเข้าโมเดล
    sampled_rows_per_source = sampled_metadata.groupby("source_dataset")["record_id"].transform("size")
    sampled_metadata["sample_weight"] = 1.0 / sampled_rows_per_source.astype(float)
    return (sampled_features, sampled_labels, sampled_metadata), sample_size

## 7. รวมข้อมูล บันทึก split และสร้างรายงาน

เซลล์นี้เรียกขั้นตอนด้านบนตามลำดับ เขียนตาราง `features`, `labels`, `metadata` แยกกัน พร้อมรายงาน readiness, ความเข้ากันได้ของ feature, จำนวนแถวแต่ละ split และรายการ KDDTest-21 ที่เชื่อมกับ Test.

In [ ]:
def build_dataset(seed: int = 42, validation_fraction: float = 0.20) -> dict:
    feature_parts, label_parts, metadata_parts, included, readiness = _read_ready_sources()
    features = pd.concat(feature_parts, ignore_index=True)
    labels = pd.concat(label_parts, ignore_index=True)
    metadata = pd.concat(metadata_parts, ignore_index=True)
    validate_aligned_tables(features, labels, metadata)
    metadata = metadata.copy()
    metadata["data_split"] = assign_splits(metadata, seed=seed, validation_fraction=validation_fraction).to_numpy()
    rows_per_source_split = metadata.groupby(["source_dataset", "data_split"])["record_id"].transform("size")
    metadata["sample_weight"] = 1.0 / rows_per_source_split.astype(float)
    root = OUTPUT_ROOT
    _write_triplet(root / "harmonized", features, labels, metadata)

    split_tables = {}
    for split in SPLITS:
        mask = metadata["data_split"].eq(split).to_numpy()
        split_tables[split] = (features.loc[mask].reset_index(drop=True), labels.loc[mask].reset_index(drop=True), metadata.loc[mask].reset_index(drop=True))
    full_train = split_tables["train"]
    train_sample, per_source = _sample_training_equally_by_source(full_train, seed)

    _write_triplet(root / "splits" / "train_full", *full_train)
    _write_triplet(root / "splits" / "train", *train_sample)
    for split in ("validation", "test"):
        _write_triplet(root / "splits" / split, *split_tables[split])

    # KDDTest-21 is a view over the official test rows, never an appended duplicate dataset.
    test_meta = split_tables["test"][2]
    reports = root / "reports"
    reports.mkdir(parents=True, exist_ok=True)
    eval_ids = test_meta.loc[test_meta["is_kddtest21_view"].astype(bool), ["record_id", "orig_row_id"]]
    eval_ids.to_csv(reports / "kddtest21_evaluation_view.csv", index=False)

    feature_status = _feature_compatibility(features, metadata, included, readiness)

    loso = [{"held_out_dataset": test_source,
             "training_datasets": [source for source in included if source != test_source],
             "status": "ready" if len(included) > 1 else "not_ready_single_source"}
            for test_source in included]
    pd.DataFrame(readiness).to_csv(reports / "dataset_readiness.csv", index=False)
    pd.DataFrame(feature_status).to_json(reports / "feature_compatibility.json", orient="records", indent=2, force_ascii=False)
    (reports / "leave_one_dataset_out.json").write_text(json.dumps(loso, indent=2, ensure_ascii=False), encoding="utf-8")
    # รายงานทั้ง Train หลังสุ่ม (ไฟล์ที่ใช้เป็นค่าเริ่มต้น) และ Train เต็มก่อนสุ่ม
    report_splits = {
        "train": train_sample[2],
        "train_full": full_train[2],
        "validation": split_tables["validation"][2],
        "test": split_tables["test"][2],
    }
    split_counts = pd.concat(
        [
            frame.groupby("source_dataset").size().rename("rows").reset_index().assign(data_split=name)
            for name, frame in report_splits.items()
        ],
        ignore_index=True,
    )[["source_dataset", "data_split", "rows"]]
    split_counts.to_csv(reports / "split_counts.csv", index=False)
    summary = {
        "included_datasets": included,
        "joint_profile_ready": set(included) == set(DATASETS),
        "missing_or_not_ready": [row["dataset"] for row in readiness if not row["ready"]],
        "rows_total": len(metadata),
        "rows_train_full": len(full_train[2]),
        "rows_train_sampled": len(train_sample[2]),
        "equal_rows_per_source_in_training": per_source,
        "sampling_seed": seed,
        "sample_weight_is_metadata_only": True,
        "split_counts": split_counts.to_dict(orient="records"),
        "kddtest21_rows_linked_to_test": len(eval_ids),
        "features_usable_in_current_build": [item["feature"] for item in feature_status if item["usable_in_current_build"]],
        "features_eligible_for_joint_profile": [item["feature"] for item in feature_status if item["eligible_for_joint_profile"]],
        "warning": ("Both configured datasets are present and validated." if set(included) == set(DATASETS)
                    else "This is a source-local/partial diagnostic build, not a verified NSL-KDD + UNSW-NB15 corpus; inspect dataset_readiness.csv.")
    }
    (reports / "build_report.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    return summary

## 8. เริ่มสร้างผลลัพธ์

รันเซลล์นี้หลังจากรันเซลล์เตรียมโค้ดทั้งหมดแล้ว ผลลัพธ์จะอยู่ใน `data-pipeline/common/output/` หาก dataset บางชุดไม่มีหรือยังไม่ผ่าน provenance ให้อ่าน `reports/dataset_readiness.csv` ก่อนนำผลไปใช้งาน.

In [ ]:
SEED = 42
VALIDATION_FRACTION = 0.20

build_summary = build_dataset(seed=SEED, validation_fraction=VALIDATION_FRACTION)
display(pd.Series(build_summary, name="ผลการสร้าง Dataset กลาง").to_frame())